# Meme Organizer: 03. Hybrid Retrieval & Cross-Encoder Re-ranking

Финальные эксперименты текстового домена поиска:
1. **First-Stage Late Fusion**:
   - **Reciprocal Rank Fusion**: объединение BM25 и Fine-tuned E5 без параметров ($k=60$).
   - **Optuna Convex Score Combination**: калибровка веса $\alpha \in [0, 1]$ на валидационном сплите с последующей честной оценкой на тесте
2. **Second-Stage Re-ranking**:
   - Отбор топ-20 кандидатов первой стадией и повторное ранжирование легковесным ONNX INT8 Cross-Encoder (`rubert-tiny-bilingual-nli`, размер 11.3 МБ).
   - Оценка выигрыша по качеству ($Recall@1$, $Recall@5$, $MRR@5$) и расходов по задержке ($ms$)


In [1]:
import json
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd

from meme_organizer.dataset.loader import DatasetLoader
from meme_organizer.models.hybrid import HybridRetriever
from meme_organizer.pipeline.retrieval_pipeline import RetrievalPipeline

GOLD_DATASET_PATH = Path("../data/gold/gold_dataset.jsonl")
SPLIT_PATH = Path("../data/splits/splits_v1.json")

2026/11/24 21:58:20 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /home/alex/Projects/meme-organizer/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## 1. Проверка золотых сплитов

In [2]:
split_info = DatasetLoader.load_split_info(SPLIT_PATH)
print(f"Корпус мемов: {len(split_info['corpus_hashes'])} документов")
print(f"Train пары:   {len(split_info['train_pairs'])} запросов")
print(f"Val пары:     {len(split_info['val_pairs'])} запросов")
print(f"Test пары:    {len(split_info['test_pairs'])} запросов")

Корпус мемов: 318 документов
Train пары:   1021 запросов
Val пары:     138 запросов
Test пары:    153 запросов


## 2. Эксперимент 1: Late Fusion via Reciprocal Rank Fusion (RRF)
RRF стабилен к разнице шкал и распределений плотных и разреженных скоров:
$$RRF(d) = \sum_{m} \frac{1}{60 + rank_m(d)}$$


In [3]:
pipeline = RetrievalPipeline()
retriever_rrf = HybridRetriever(mode="rrf", model_name="hybrid-rrf")

metrics_rrf, run_id_rrf = pipeline.run(
    retriever=retriever_rrf,
    gold_dataset_path=GOLD_DATASET_PATH,
    split_path=SPLIT_PATH,
    split_name="test",
)

pd.DataFrame(
    [
        {
            "Model": "Hybrid RRF (BM25 + E5-FT)",
            "Recall@1": metrics_rrf.recall_at_1,
            "Recall@5": metrics_rrf.recall_at_5,
            "MRR@5": metrics_rrf.mrr_at_5,
            "NDCG@5": metrics_rrf.ndcg_at_5,
            "MLflow Run ID": run_id_rrf,
        }
    ]
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🏃 View run hybrid-rrf_test at: http://localhost:5000/#/experiments/6/runs/14cba2d104934537b67beb887adf6752
🧪 View experiment at: http://localhost:5000/#/experiments/6


,Model,Recall@1,Recall@5,MRR@5,NDCG@5,MLflow Run ID
0,Hybrid RRF (BM25 + E5-FT),0.6078,0.7059,0.6444,0.6598,14cba2d104934537b67beb887adf6752


## 3. Эксперимент 2: Convex Score Combination с подбором веса $\alpha$ через Optuna
Подбор веса $\alpha$ происходит **строго на валидационном сплите** (`val_pairs`), после чего замороженный вес применяется к тесту (`test_pairs`).


In [4]:
retriever_optuna = HybridRetriever(mode="optuna", model_name="hybrid-optuna")

metrics_optuna, run_id_optuna = pipeline.run(
    retriever=retriever_optuna,
    gold_dataset_path=GOLD_DATASET_PATH,
    split_path=SPLIT_PATH,
    split_name="test",
)

print(f"Подобранный вес alpha (Dense vs BM25): {retriever_optuna.alpha:.4f}")
pd.DataFrame(
    [
        {
            "Model": f"Hybrid Convex (alpha={retriever_optuna.alpha:.2f})",
            "Recall@1": metrics_optuna.recall_at_1,
            "Recall@5": metrics_optuna.recall_at_5,
            "MRR@5": metrics_optuna.mrr_at_5,
            "NDCG@5": metrics_optuna.ndcg_at_5,
            "MLflow Run ID": run_id_optuna,
        }
    ]
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🏃 View run hybrid-optuna_test at: http://localhost:5000/#/experiments/6/runs/d5ee8d491cf042b5b4805f472f6f58fc
🧪 View experiment at: http://localhost:5000/#/experiments/6
Подобранный вес alpha (Dense vs BM25): 0.1050


,Model,Recall@1,Recall@5,MRR@5,NDCG@5,MLflow Run ID
0,Hybrid Convex (alpha=0.11),0.6275,0.7582,0.6785,0.6985,d5ee8d491cf042b5b4805f472f6f58fc


## 4. Эксперимент 3: Two-Stage Retrieval с Cross-Encoder Re-ranking
Отбираем топ-20 кандидатов первой стадией и реранжируем их с помощью `rubert-tiny-bilingual-nli` (INT8 ONNX).


In [5]:
retriever_ce = HybridRetriever(
    mode="rerank_ce",
    model_name="hybrid-rerank-ce",
    rerank_top_k=20,
    alpha=retriever_optuna.alpha,
)

metrics_ce, run_id_ce = pipeline.run(
    retriever=retriever_ce,
    gold_dataset_path=GOLD_DATASET_PATH,
    split_path=SPLIT_PATH,
    split_name="test",
)

pd.DataFrame(
    [
        {
            "Model": "Two-Stage (Hybrid + CE Reranker)",
            "Recall@1": metrics_ce.recall_at_1,
            "Recall@5": metrics_ce.recall_at_5,
            "MRR@5": metrics_ce.mrr_at_5,
            "NDCG@5": metrics_ce.ndcg_at_5,
            "MLflow Run ID": run_id_ce,
        }
    ]
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🏃 View run hybrid-rerank-ce_test at: http://localhost:5000/#/experiments/6/runs/e7accbb781444cf98a1e0e8999b9f420
🧪 View experiment at: http://localhost:5000/#/experiments/6


,Model,Recall@1,Recall@5,MRR@5,NDCG@5,MLflow Run ID
0,Two-Stage (Hybrid + CE Reranker),0.4837,0.719,0.5781,0.6137,e7accbb781444cf98a1e0e8999b9f420


## 5. Итоговый лидерборд текстового поиска

In [6]:
client = mlflow.tracking.MlflowClient(tracking_uri="http://localhost:5000")
exp = client.get_experiment_by_name("meme-organizer-retrieval")
runs = client.search_runs(experiment_ids=[exp.experiment_id], order_by=["attributes.start_time ASC"])

data = []
for r in runs:
    name = r.data.tags.get("mlflow.runName", "unnamed")
    m = r.data.metrics
    data.append(
        {
            "Model": name,
            "Recall@1": m.get("recall_at_1", 0),
            "Recall@5": m.get("recall_at_5", 0),
            "MRR@5": m.get("mrr_at_5", 0),
            "NDCG@5": m.get("ndcg_at_5", 0),
            "P95 Latency (ms)": m.get("cpu_latency_p95_ms", 0),
            "Model Size (MB)": m.get("model_size_mb", 0),
            "Eval Time (s)": m.get("eval_inference_time_sec", 0),
        }
    )

df_leaderboard = pd.DataFrame(data)
df_leaderboard.sort_values(by="MRR@5", ascending=False).reset_index(drop=True)

,Model,Recall@1,Recall@5,MRR@5,NDCG@5,P95 Latency (ms),Model Size (MB),Eval Time (s)
0,test-bm25_test,0.5000,1.0000,0.7500,0.8155,0.03,0.10,0.000
1,hybrid-optuna_test,0.6275,0.7582,0.6785,0.6985,22.67,112.62,3.167
2,bm25-baseline_test,0.6078,0.7255,0.6544,0.6722,0.10,0.10,0.018
3,hybrid-rrf_test,0.6078,0.7059,0.6444,0.6598,11.62,112.62,2.736
4,multilingual-e5-small-finetuned_test,0.5817,0.6863,0.6208,0.6371,7.77,112.52,3.174
5,multilingual-e5-small_test,0.5686,0.6797,0.6132,0.6300,7.67,112.52,2.688
6,hybrid-rerank-ce_test,0.4837,0.7190,0.5781,0.6137,76.51,123.94,10.381
7,rubert-tiny2_test,0.4248,0.5033,0.4517,0.4645,1.49,27.93,0.378
8,rubert-tiny2-finetuned_test,0.4052,0.5098,0.4447,0.4610,3.00,27.93,0.442
9,hybrid-rrf_test,0.0000,0.0000,0.0000,0.0000,0.00,0.00,0.000
